In [946]:
#Import the library
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [948]:
#Define the column names based on the UCI documentation
column_names = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num', 
    'marital-status', 'occupation', 'relationship', 'race', 'sex', 
    'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 
    'income'
]

#Inject the column name in the dataset
dataset = pd.read_csv('adult.data', names = column_names,skipinitialspace=True )

# Splitting the data into independent variable X and dependent variable y
X = dataset.iloc[:,:-1].values
y = dataset.iloc[:,-1].values

In [950]:
dataset.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [952]:
X

array([[39, 'State-gov', 77516, ..., 0, 40, 'United-States'],
       [50, 'Self-emp-not-inc', 83311, ..., 0, 13, 'United-States'],
       [38, 'Private', 215646, ..., 0, 40, 'United-States'],
       ...,
       [58, 'Private', 151910, ..., 0, 40, 'United-States'],
       [22, 'Private', 201490, ..., 0, 20, 'United-States'],
       [52, 'Self-emp-inc', 287927, ..., 0, 40, 'United-States']],
      dtype=object)

In [954]:
y

array(['<=50K', '<=50K', '<=50K', ..., '<=50K', '<=50K', '>50K'],
      dtype=object)

In [956]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             32561 non-null  int64 
 1   workclass       32561 non-null  object
 2   fnlwgt          32561 non-null  int64 
 3   education       32561 non-null  object
 4   education-num   32561 non-null  int64 
 5   marital-status  32561 non-null  object
 6   occupation      32561 non-null  object
 7   relationship    32561 non-null  object
 8   race            32561 non-null  object
 9   sex             32561 non-null  object
 10  capital-gain    32561 non-null  int64 
 11  capital-loss    32561 non-null  int64 
 12  hours-per-week  32561 non-null  int64 
 13  native-country  32561 non-null  object
 14  income          32561 non-null  object
dtypes: int64(6), object(9)
memory usage: 3.7+ MB


In [958]:
#Check if there is any null values in the dataset
dataset.isnull().values.any()

np.False_

In [960]:
#CHecking how many unique values are there in  column workclass
dataset['workclass'].unique()

array(['State-gov', 'Self-emp-not-inc', 'Private', 'Federal-gov',
       'Local-gov', '?', 'Self-emp-inc', 'Without-pay', 'Never-worked'],
      dtype=object)

In [962]:
# Splitting the data into training and test set, this is done before applying Categorical Encoding to avoid memory leakage
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.2, random_state = 0)

In [964]:
X_train

array([[36, 'Private', 174308, ..., 0, 40, 'United-States'],
       [35, 'Private', 198202, ..., 0, 54, 'United-States'],
       [38, 'Private', 52963, ..., 0, 50, 'United-States'],
       ...,
       [23, 'Private', 45317, ..., 0, 40, 'United-States'],
       [45, 'Local-gov', 215862, ..., 0, 45, 'United-States'],
       [25, 'Private', 186925, ..., 0, 48, 'United-States']], dtype=object)

In [966]:
y_train

array(['<=50K', '<=50K', '<=50K', ..., '<=50K', '>50K', '<=50K'],
      dtype=object)

In [968]:
type(X_train)

numpy.ndarray

In [970]:
# Convert X_train and X_test to DataFrame
# Get all the column name from Dataframe except for the dependent variable y column name
column_names_X = dataset.drop('income', axis=1).columns.tolist()
#Converting X_train and X_test to DF since ColumnTransformer fit_transform will accept only DF and no numpy array
X_train = pd.DataFrame(X_train, columns=column_names_X)
X_test  = pd.DataFrame(X_test,  columns=column_names_X)

In [974]:
#Check if there are any missing values of special characters. 
# As mentioned in the document, Convert Unknown to "?" (Unknown values were converted to ?)
# These ? values should be converted to null and then replaced with the mean if it is numeric column
# Categorical column should be converted to 
# Checking X_train first
print(X_train.isnull().any())
print(X_test.isnull().any())

age               False
workclass         False
fnlwgt            False
education         False
education-num     False
marital-status    False
occupation        False
relationship      False
race              False
sex               False
capital-gain      False
capital-loss      False
hours-per-week    False
native-country    False
dtype: bool
age               False
workclass         False
fnlwgt            False
education         False
education-num     False
marital-status    False
occupation        False
relationship      False
race              False
sex               False
capital-gain      False
capital-loss      False
hours-per-week    False
native-country    False
dtype: bool


In [976]:
type(X_train)

pandas.core.frame.DataFrame

In [980]:
# The three columns with ? values are all Categorical column in X_train and X_test, they should be first converted to Nan
import numpy as np
X_train = pd.DataFrame(X_train).replace('?', np.nan).infer_objects(copy=False)
X_test = pd.DataFrame(X_test).replace('?', np.nan).infer_objects(copy=False)

In [982]:
# Using Simple Imputer to convert Null values to most frequent value
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(missing_values = np.nan, strategy='most_frequent')
X_train = imputer.fit_transform(X_train)
X_test  = imputer.transform(X_test) 

In [984]:
type(X_train)

numpy.ndarray

In [986]:
# Check again if there are any null or ? values in X_train and X_test
print(pd.DataFrame(X_train).isnull().sum().sum())
print(pd.DataFrame(X_test).isnull().sum().sum())

0
0


In [988]:
type(X_train)

numpy.ndarray

In [990]:
# Convert X_train and X_test back to DataFrame, this is done since Columntransformer fit transform method accepts Dataframe
# Get all the column name from Dataframe except for the dependent variable y column name
column_names_X = dataset.drop('income', axis=1).columns.tolist()
#Converting X_train and X_test to DF since ColumnTransformer fit_transform will accept only DF and no numpy array
X_train = pd.DataFrame(X_train, columns=column_names_X)
X_test  = pd.DataFrame(X_test,  columns=column_names_X)

In [766]:
# From the output above there are no ? or null values,out data is ready for Categorical Encoding and Scaling

In [992]:
# Applying Categorical Ecoding using OneHotEncoder and Columntransformer
# Also applying Standard Scalar to the numerical value
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer

# These are the columns in independent variable that has categorical data
catergorical_columns = ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 
                        'race', 'sex','native-country']
# These are the columns with Numerical Data
scalar_columns = ['age','fnlwgt','education-num','capital-gain','capital-loss','hours-per-week']
ct = ColumnTransformer(transformers = [('scalar',StandardScaler(),scalar_columns),
                      ('encoder',OneHotEncoder(drop = 'first',handle_unknown='ignore', sparse_output=False),catergorical_columns)],remainder = 'passthrough')
X_train_encoded = ct.fit_transform(X_train)
# Training the independent test data for Categorical Encoding
X_test_encoded = ct.transform(X_test)

In [994]:
# All the categorical data is now converted to 0s and 1s
X_train_encoded

array([[-0.18928087, -0.14399738, -1.19288369, ...,  1.        ,
         0.        ,  0.        ],
       [-0.26265957,  0.08271497, -0.41510561, ...,  1.        ,
         0.        ,  0.        ],
       [-0.04252347, -1.2953496 ,  1.14045055, ...,  1.        ,
         0.        ,  0.        ],
       ...,
       [-1.14320394, -1.36789679, -0.02621657, ...,  1.        ,
         0.        ,  0.        ],
       [ 0.47112741,  0.25027755,  2.30711767, ...,  1.        ,
         0.        ,  0.        ],
       [-0.99644654, -0.02428407, -0.02621657, ...,  1.        ,
         0.        ,  0.        ]])

In [996]:
X_train_encoded.shape

(26048, 97)

In [998]:
#Label Encoding the dependent Variable, it has the value <50k or >50k. 
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

In [1000]:
y_train_encoded

array([0, 0, 0, ..., 0, 1, 0])

In [1002]:
y_train_encoded[0:20]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0])

In [1004]:
y_train[0:20]

array(['<=50K', '<=50K', '<=50K', '<=50K', '<=50K', '<=50K', '<=50K',
       '<=50K', '<=50K', '<=50K', '<=50K', '<=50K', '<=50K', '<=50K',
       '<=50K', '>50K', '<=50K', '<=50K', '<=50K', '<=50K'], dtype=object)

In [1006]:
y_train_encoded


array([0, 0, 0, ..., 0, 1, 0])

In [266]:
# Training Data
# X_train_encoded
# y_train_encoded
#Testing Data
# X_test_encoded
# y_test_encoded

In [1008]:
# 1) Applying the first Classification model Logistic Regression Model
from sklearn.linear_model import LogisticRegression
classifier = LogisticRegression()
classifier.fit(X_train_encoded,y_train_encoded)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [1010]:
# Prediting the value of test data
y_pred = classifier.predict(X_test_encoded)

In [1012]:
print(np.concatenate((y_pred.reshape(len(y_pred),1),y_test_encoded.reshape(len(y_test_encoded),1)),1))

[[0 0]
 [0 0]
 [0 0]
 ...
 [1 1]
 [0 0]
 [0 1]]


In [1016]:
#Finding the accuracy and making the confusion matrix
from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(y_test_encoded,y_pred)
print(cm)
accuracy_score(y_test_encoded,y_pred)

[[4551  367]
 [ 633  962]]


0.8464609243052357

In [ ]:
# Logistic Regression has given the accuracy score of 84.64%, which is pretty good for a linear Model

In [1018]:
#Implementing K - Nearest Neighbors
# Training the model
from sklearn.neighbors import KNeighborsClassifier
classifier = KNeighborsClassifier(n_neighbors = 5, metric = 'minkowski',p=2)
classifier.fit(X_train_encoded,y_train_encoded)

,n_neighbors,5
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [1020]:
# Predicting the test result
y_pred = classifier.predict(X_test_encoded)

In [1022]:
#Finding the accuracy and making the confusion matrix
from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(y_test_encoded,y_pred)
print(cm)
accuracy_score(y_test_encoded,y_pred)

[[4428  490]
 [ 607  988]]


0.8315676339628435

In [282]:
# KNN has accuracy score of 83.16%, which is less than LinearRegression model

In [1024]:
#Implementing Support Vector Machine SVM Model
from sklearn.svm import SVC
classifier = SVC(kernel = 'linear',random_state = 0)
classifier.fit(X_train_encoded,y_train_encoded)

,C,1.0
,kernel,'linear'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [1026]:
# Predicting the test result
y_pred = classifier.predict(X_test_encoded)

In [1028]:
#Finding the accuracy and making the confusion matrix
from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(y_test_encoded,y_pred)
print(cm)
accuracy_score(y_test_encoded,y_pred)

[[4597  321]
 [ 670  925]]


0.8478427759864886

In [287]:
# SVM has accuracy of 84.7% which is better than LR and KNN

In [1030]:
#Predicting the output using kernel SVM
from sklearn.svm import SVC
classifier = SVC(kernel = 'rbf')
classifier.fit(X_train_encoded,y_train_encoded)

,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [1032]:
#Predicting the test result
y_pred = classifier.predict(X_test_encoded)

In [1034]:
#Finding the accuracy and making the confusion matrix
from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(y_test_encoded,y_pred)
print(cm)
accuracy_score(y_test_encoded,y_pred)

[[4610  308]
 [ 659  936]]


0.8515277138031629

In [ ]:
# Accuracy is almost same as SVM, it has moved from 84.7 to 85.15%
# rbf accuracy 85.15%
# poly accuracy is slow, accuracy is 84.8%

In [1036]:
# Training the Naive Bayes model on the Training set
from sklearn.naive_bayes import GaussianNB
classifier = GaussianNB()
classifier.fit(X_train_encoded,y_train_encoded)

,priors,None
,var_smoothing,1e-09


In [1038]:
# Predicting the test result set
y_pred = classifier.predict(X_test_encoded)

In [1040]:
#Finding the accuracy and making the confusion matrix
from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(y_test_encoded,y_pred)
print(cm)
accuracy_score(y_test_encoded,y_pred)

[[1982 2936]
 [  62 1533]]


0.5396898510670965

In [ ]:
# Naive bayes has very low accuracy of 54%, the reason is Naive Bayes assumes all features are completely independent of each other.
# But in this Adult Income Dataset all the features are heavily connected. NB is not a good model for this DataSet

In [1042]:
# Training the Decision Tree Classification model on the Training set
from sklearn.tree import DecisionTreeClassifier
classifier = DecisionTreeClassifier(criterion = 'entropy', random_state = 0)
classifier.fit(X_train_encoded,y_train_encoded)

,criterion,'entropy'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [1044]:
# Predicting the value of test data
y_pred = classifier.predict(X_test_encoded)

In [1046]:
#Finding the accuracy score and making the confusion matrix of Decision Tree
from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(y_test_encoded,y_pred)
print(cm)
accuracy_score(y_test_encoded,y_pred)

[[4284  634]
 [ 608  987]]


0.8093044679871028

In [ ]:
#Decision tree perfomance is not better than others, with accuracy score of 80.93%

In [1048]:
#Training the Random Forest Classification model on the Training set
from sklearn.ensemble import RandomForestClassifier
classifier = RandomForestClassifier(n_estimators=200, criterion='entropy')
classifier.fit(X_train_encoded,y_train_encoded)

,n_estimators,200
,criterion,'entropy'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [1050]:
#Predicting the test result
y_pred = classifier.predict(X_test_encoded)

In [1052]:
#Finding the accuracy score and making the confusion matrix of Random Forest Regression
from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(y_test_encoded,y_pred)
print(cm)
accuracy_score(y_test_encoded,y_pred)

[[4553  365]
 [ 611  984]]


0.85014586212191

In [ ]:
# Accuracy of 84.4% with Random Forest Algoright ,
#n_estimators=50 , accuracy score of 84.8%
#n_estimators=100, accuracy score of 84.9%
#n_estimators=150, accuracy score of 85.3%
#n_estimators=200, accuracy score of 84.7
# n_estimators=150, accuracy score of 85.3% is the best accuracy provided by Random Forest Regression

In [1054]:
# Training XGBoost on the Training set
from xgboost import XGBClassifier
classifier = XGBClassifier()
classifier.fit(X_train_encoded,y_train_encoded)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [1056]:
#Finding the accuracy score and making the confusion matrix of XGBoost
from sklearn.metrics import confusion_matrix, accuracy_score
y_pred = classifier.predict(X_test_encoded)
cm = confusion_matrix(y_test_encoded, y_pred)
print(cm)
accuracy_score(y_test_encoded, y_pred)

[[4574  344]
 [ 536 1059]]


0.8648856133886074

In [ ]:
#XGboost has the best accuracy score of 86.49%
#In the next set of code, I am tuning the paramters to see if higher accuracy can be achieved

In [1058]:
# Training XGBoost on the Training set with parameter tuning
from xgboost import XGBClassifier
classifier = XGBClassifier( n_estimators = 75,
         learning_rate     = 0.3,
         max_depth         = 6)
         # subsample         = 0.9,
         # colsample_bytree  = 0.8)
      
classifier.fit(X_train_encoded,y_train_encoded)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [1060]:
#Finding the accuracy score and making the confusion matrix of XGBoost
from sklearn.metrics import confusion_matrix, accuracy_score
y_pred = classifier.predict(X_test_encoded)
cm = confusion_matrix(y_test_encoded, y_pred)
print(cm)
accuracy_score(y_test_encoded, y_pred)

[[4589  329]
 [ 534 1061]]


0.8674957776754184

In [ ]:
#  n_estimators = 75, learning_rate     = 0.3, max_depth  = 6 accuracy of 86.751%
# And this is the best accuracy XGBoost is the winner